<a href="https://colab.research.google.com/github/zohrehasadi00/automatic_image_analysis/blob/main/notebooks/Modern_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 🛠 Troubleshooting: GitHub Rendering Error
If you see the error `the 'state' key is missing from 'metadata.widgets'` on GitHub, follow these steps:
1. Go to **Edit** -> **Notebook settings**.
2. Ensure **"Omit code cell output when saving this notebook"** is unchecked.
3. **Recommended:** Before saving to GitHub, go to **Runtime** -> **Clear all outputs**. This removes the widget metadata that causes the rendering error.

#Fundamental setup

In [1]:
# import collection

import os
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import torch.optim as optim
from tqdm.auto import tqdm
import torch.nn.functional as F

from scipy.sparse import find

In [2]:
#set seed for reproducibility
random_state = 42
torch.manual_seed(random_state)
np.random.seed(random_state)


In [3]:
# connect Drive for data
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# set project_dir
project_dir = Path('.') / 'drive' / 'MyDrive' / 'AIA_SkinLesion_Projekt'

In [5]:
#path to image data
data_raw =[
    str(project_dir / 'data' / 'raw' / 'HAM10000_images_part_1'),
    str(project_dir / 'data' / 'raw' / 'HAM10000_images_part_2'),
    ]

# load metadata
csv_path    = project_dir / 'HAM10000_metadata_converted.csv'
df_metadata = pd.read_csv(csv_path)

#load splits
csv_train = project_dir / 'splits' / 'train.csv'
df_train  = pd.read_csv(csv_train)

csv_train_balanced = project_dir / 'splits' / 'train_balanced.csv'
df_train_balanced  = pd.read_csv(csv_train_balanced)

csv_test = project_dir / 'splits' / 'test.csv'
df_test  = pd.read_csv(csv_test)

csv_val = project_dir / 'splits' / 'val.csv'
df_val  = pd.read_csv(csv_val)

# Data preprocessing for modern pipeline

In [6]:
# EfficientNet-B0 expects 224x224; other resolutions are possible; includes ImageNet normalization

# define image transformations for train and test/validation set
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(), # avoid overfitting in training data
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

## function for searching images from drive
def find_image_path(img_id, directories):
    img_name = img_id + '.jpg'
    for directory in directories:
        path = os.path.join(directory, img_name)
        if os.path.exists(path):
            return path
    return None

# Dataset class
class SkinDataset(Dataset):
    def __init__(self, dataframe, img_dirs, transform=None):
        self.data = dataframe.reset_index(drop=True)
        self.img_dirs = img_dirs
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_id = self.data.loc[idx, 'image_id']
        img_name = img_id + '.jpg'
        img_path = find_image_path(img_id, self.img_dirs)

        if img_path is None:
            raise FileNotFoundError(f"Image {img_name} not found!")

        image = Image.open(img_path).convert('RGB')
        label = self.data.loc[idx, 'label']

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.float32), img_id

# Use balanced training set if loaded, otherwise use standard df_train
train_set_to_use = df_train_balanced if 'df_train_balanced' in locals() else df_train

# create data sets
train_dataset = SkinDataset(train_set_to_use, data_raw, transform=train_transforms)
val_dataset = SkinDataset(df_val, data_raw, transform=test_transforms)
test_dataset = SkinDataset(df_test, data_raw, transform=test_transforms)

# create data loader
b_size = 32
train_loader = DataLoader(train_dataset, batch_size=b_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=b_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=b_size, shuffle=False)

print(f"Dataset ready. Training uses: {'Balanced Set' if 'df_train_balanced' in locals() else 'Standard Set'}")
print(f"Number of training images: {len(train_dataset)}")

Dataset ready. Training uses: Balanced Set
Number of training images: 2300


In [ ]:
### Example output
images, labels, image_ids = next(iter(train_loader))


tensor_img = images[-1]
label = labels[-1]
last_id = image_ids[-1]  # Das ist z.B. "ISIC_0024306"

print(f"Loading original image with id: {last_id}.jpg ...")

# image after transformation
img_np = tensor_img.numpy().transpose((1, 2, 0))
#mean = np.array([0.485, 0.456, 0.406])
#std = np.array([0.229, 0.224, 0.225])
#bild_np = std * bild_np + mean
minimum = img_np.min()
maximum = img_np.max()
img_np = (img_np - minimum) / (maximum - minimum)

# original image
img_name = last_id + '.jpg'
original_path = find_image_path(last_id, data_raw)

# show images
if original_path:
    original_image = Image.open(original_path).convert('RGB')
    label_name = "malignant (1)" if label.item() == 1.0 else "benign (0)"

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    fig.suptitle(f"Control check: last image in batch\nid: {last_id} | diagnosis: {label_name}", fontsize=14, fontweight='bold')

    axes[0].imshow(original_image)
    axes[0].set_title(f"before \nfile name: {img_name}\nsize: {original_image.size}", fontsize=12)
    axes[0].axis('off')

    axes[1].imshow(img_np)
    axes[1].set_title(f"after \nsize: {tensor_img.shape[1]}x{tensor_img.shape[2]}", fontsize=12)
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()
else:
    print(f"Error: {img_name} could not be found!")

check_row = df_metadata[df_metadata['image_id'] == last_id]

print(f"Searched for id: {last_id}")
print(f"ID from csv:\n{check_row[['image_id', 'label']]}")

# additional check
csv_label = check_row['label'].values[0]
batch_label = int(label.item())

if csv_label == batch_label:
    print("\n Matches csv")
else:
    print("\n Differs from csv")

print("finished")

```markdown
# Modern Pipeline: EfficientNet-B0

In this section, we implement the EfficientNet-B0 model.
- The feature extractor is frozen (**frozen weights**).
- The classification head is adapted for binary classification.
- For uncertainty estimation, we implement a logic for **Monte Carlo (MC) Dropout**.
```

In [7]:
def get_efficientnet_model(dropout_rate=0.5):
    # 1. Load pre-trained EfficientNet-B0 weights from ImageNet
    model = models.efficientnet_b0(weights='IMAGENET1K_V1')

    # 2. Freeze backbone: Prevent weights from updating during training
    for param in model.parameters():
        param.requires_grad = False

    # 3. Adapt classification head for binary output (1 neuron)
    num_ftrs = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout_rate, inplace=True),
        nn.Linear(num_ftrs, 1)
    )

    return model

# Initialize model and move to GPU/CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = get_efficientnet_model()
model = model.to(device)

print(f"Model loaded on {device}.")
# Check trainable params: should only be the new classifier head
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters (head only): {trainable_params}")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 129MB/s] 


Model loaded on cpu.
Trainable parameters (head only): 1281


```markdown
### Note on MC Dropout
To measure uncertainty later, we must ensure that Dropout remains active even during the test phase (`model.eval()`). We achieve this by manually setting the Dropout layers back to `.train()`.
```

In [8]:
def enable_dropout(m):
    """
    Forces Dropout layers to stay in .train() mode even during evaluation.
    This allows for Monte Carlo (MC) sampling to estimate uncertainty.
    """
    for module in m.modules():
        if isinstance(module, nn.Dropout):
            module.train()

```markdown
## Training and Validation Setup
We define the loss function (BCEWithLogitsLoss), the optimizer (Adam), and a loop to train and validate the model across several epochs.
```

In [11]:
# --- SETTINGS ---
num_epochs = 1
learning_rate = 0.001
# TIP: Set limit_batches to 1 for a quick test run. Set to None for full training.
limit_batches = 10
save_weights = False # FLAG: Set to True only when you want to export the model file

# 2. Loss and Optimizer setup
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=learning_rate)

# 3. Training Loop
train_losses = []
val_losses = []

print(f"Starting training on {device}...")

for epoch in range(num_epochs):
    # --- TRAINING PHASE ---
    model.train() #does mc dropout
    running_loss = 0.0

    # Removed tqdm for GitHub compatibility
    for i, (images, labels, _) in enumerate(train_loader):
        if limit_batches and i >= limit_batches: break

        images, labels = images.to(device), labels.to(device).unsqueeze(1)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_train_loss = running_loss / (len(train_loader.dataset) if not limit_batches else (limit_batches * b_size))
    train_losses.append(epoch_train_loss)

    # --- VALIDATION PHASE ---
    model.eval()
    running_val_loss = 0.0

    with torch.no_grad():
        for i, (images, labels, _) in enumerate(val_loader):
            if limit_batches and i >= limit_batches: break

            images, labels = images.to(device), labels.to(device).unsqueeze(1)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item() * images.size(0)

    epoch_val_loss = running_val_loss / (len(val_loader.dataset) if not limit_batches else (limit_batches * b_size))
    val_losses.append(epoch_val_loss)

    print(f"Epoch [{epoch+1}/{num_epochs}] completed. Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")

# 4. Optional Saving
if save_weights:
    save_path = project_dir / 'models' / 'efficientnet_skin_model_weights.pth'
    os.makedirs(save_path.parent, exist_ok=True)
    torch.save(model.state_dict(), save_path)
    print(f"Model saved to {save_path}")
else:
    print("Run finished! Weights were kept in RAM only.")

Starting training on cpu...
Epoch [1/1] completed. Train Loss: 0.6352 | Val Loss: 0.6500
Run finished! Weights were kept in RAM only.


In [ ]:
import matplotlib.pyplot as plt

# Visualize training and validation loss
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

### 📂 Loading Saved Weights
Use this block if you want to load a previously trained model from your Google Drive instead of running a new training.

In [ ]:
# 1. Define file path
weights_path = project_dir / 'models' / 'efficientnet_skin_model_weights.pth'

# 2. Check if file exists
if weights_path.exists():
    # 3. Load the weights (state_dict)
    # map_location='cpu' allows loading even without a GPU
    state_dict = torch.load(weights_path, map_location=device)

    # 4. Transfer weights into the model
    model.load_state_dict(state_dict)
    model.to(device)

    print(f"✅ Weights successfully loaded from {weights_path}.")
else:
    print(f"⚠️ No weight file found at {weights_path}. Did you use 'save_weights = True' during training?")

In [10]:


def predict_with_uncertainty(model, dataloader, num_samples=10):
    """ Performs MC Dropout by running multiple forward passes """
    model.eval()
    enable_dropout(model) # Keep Dropout active

    images, labels, ids = next(iter(dataloader))
    images = images.to(device)

    with torch.no_grad():
        # Collect predictions from multiple passes
        outputs = torch.stack([model(images) for _ in range(num_samples)])
        probs = torch.sigmoid(outputs)

    mean_probs = probs.mean(dim=0).cpu().numpy()
    std_probs = probs.std(dim=0).cpu().numpy()

    return images.cpu(), labels.numpy(), mean_probs, std_probs, ids

# --- Demonstration ---
imgs, lbls, means, stds, img_ids = predict_with_uncertainty(model, val_loader)

idx = 0
print(f"Image ID: {img_ids[idx]}")
print(f"True Label: {'Malignant' if lbls[idx] == 1 else 'Benign'}")
print(f"Model Prediction (Mean Prob): {means[idx][0]:.4f}")
print(f"Uncertainty (Std Dev): {stds[idx][0]:.4f}")

Image ID: ISIC_0029176
True Label: Benign
Model Prediction (Mean Prob): 0.5265
Uncertainty (Std Dev): 0.0547


## Final Evaluation on Test Set
Execute this cell ONLY after you have performed a full training run (with `limit_batches = None`).

In [ ]:
def evaluate_test_set(model, dataloader):
    model.eval()
    # Note: We do NOT use enable_dropout here for the standard 'final' performance,
    # unless you want the mean of MC Dropout samples as your final prediction.

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels, _ in tqdm(dataloader, desc="Evaluating Test Set"):
            images = images.to(device)
            outputs = model(images)
            preds = torch.sigmoid(outputs)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    return np.array(all_labels), np.array(all_preds)

# Run evaluation
# labels_test, preds_test = evaluate_test_set(model, test_loader)
# print("Evaluation finished. You can now calculate metrics like AUC-ROC.")

### MC Dropout Evaluation on Test Set
This cell applies MC Dropout to the entire Test Set to get uncertainty estimates for every final prediction.

In [ ]:
def evaluate_test_set_with_uncertainty(model, dataloader, num_samples=10):
    model.eval()
    enable_dropout(model) # Ensure Dropout stays active

    all_means = []
    all_stds = []
    all_labels = []
    all_ids = []

    with torch.no_grad():
        for images, labels, ids in tqdm(dataloader, desc="MC Dropout Test Evaluation"):
            images = images.to(device)

            # Multiple forward passes
            outputs = torch.stack([model(images) for _ in range(num_samples)])
            probs = torch.sigmoid(outputs)

            mean_probs = probs.mean(dim=0).cpu().numpy()
            std_probs = probs.std(dim=0).cpu().numpy()

            all_means.extend(mean_probs)
            all_stds.extend(std_probs)
            all_labels.extend(labels.numpy())
            all_ids.extend(ids)

    return np.array(all_ids), np.array(all_labels), np.array(all_means), np.array(all_stds)

# To run after full training:
# test_ids, test_labels, test_means, test_stds = evaluate_test_set_with_uncertainty(model, test_loader)
# print("MC Dropout evaluation on test set complete.")

## 🏁 Final Step: Inference on the Test Set
After your full training is complete, run this cell to get the final results for your project report.

In [ ]:
# 1. Run the evaluation on the test set
test_ids, test_labels, test_means, test_stds = evaluate_test_set_with_uncertainty(model, test_loader, num_samples=20)

#2. Convert to a DataFrame for easy analysis
results_df = pd.DataFrame({
    'image_id': test_ids,
    'true_label': test_labels,
    'prediction_prob': test_means.flatten(),
    'uncertainty_std': test_stds.flatten()
})

#3. Show first few results
display(results_df.head())

#4. Save results to CSV
results_df.to_csv(project_dir / 'final_test_results_modern.csv', index=False)
print("Final results saved to Drive!")